In [ ]:
import os
import random
import shutil

In [ ]:
# kopplar till drive för att hemta dtaset
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# kollar att datasettet ser rätt ut
!ls /content/drive/MyDrive/TrashDataset

In [ ]:
# hemtar alla bilder och lables och skriver hur många det finns

images = os.listdir("/content/drive/MyDrive/TrashDataset/images")
labels = os.listdir("/content/drive/MyDrive/TrashDataset/labels")

print("Images:", len(images))
print("Labels:", len(labels))

In [ ]:
# anger sökväg till mappen och lagar ny outputmap

base = "/content/drive/MyDrive/TrashDataset"

images_dir = f"{base}/images"
labels_dir = f"{base}/labels"

output = "/content/waste_dataset"

splits = ["train","val","test"]

# definierar splitten för dataset
for s in splits:
    os.makedirs(f"{output}/images/{s}", exist_ok=True)
    os.makedirs(f"{output}/labels/{s}", exist_ok=True)

images = os.listdir(images_dir)

# samlar endast bilder som har en tillhörande lable eftersom antalet var ojämnt
valid = []
for img in images:
    label = img.rsplit(".",1)[0] + ".txt"
    if os.path.exists(os.path.join(labels_dir,label)):
        valid.append(img)

# blandar bilderna innan uppdelning
random.shuffle(valid)

# delar upp dataset
n = len(valid)
train = valid[:int(n*0.8)]
val = valid[int(n*0.8):int(n*0.9)]
test = valid[int(n*0.9):]

dataset = {"train":train,"val":val,"test":test}

# kopierar in de sorterade bilder och lables i rätt mapp
for split,files in dataset.items():
    for img in files:
        label = img.rsplit(".",1)[0] + ".txt"

        shutil.copy(os.path.join(images_dir,img),
                    f"{output}/images/{split}/{img}")

        shutil.copy(os.path.join(labels_dir,label),
                    f"{output}/labels/{split}/{label}")

print("DONE")
print("train:",len(train))
print("val:",len(val))
print("test:",len(test))

In [ ]:
# installerar Ultralytics som används för att träna modellen

!pip install -q ultralytics

In [ ]:
# skapar em yaml fil som beskriver datasettets struktur och klasser

data_yaml = """
path: /content/waste_dataset
train: images/train
val: images/val
test: images/test

names:
  0: Dryckeskartong
  1: Konservburk
  2: Pantburk
"""

# sparar den i datasetmappen
with open("/content/waste_dataset/data.yaml", "w", encoding="utf-8") as f:
    f.write(data_yaml)

print("data.yaml skapad")

In [ ]:
!cat /content/waste_dataset/data.yaml

In [ ]:
# importera yolo och ladda ner en modell

from ultralytics import YOLO

model = YOLO("yolo26n.pt")

# träna modellen på datasettet
results = model.train(
    data="/content/waste_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    project="/content/runs",
    name="waste_yolo26"
)

In [ ]:
!ls /content/runs/waste_yolo262/weights

In [ ]:
# ladda ner den best trenade modellen

from google.colab import files
files.download("/content/runs/waste_yolo262/weights/best.pt")

In [ ]:
# ladda ner resultatet

files.download("/content/runs/waste_yolo262/results.png")
files.download("/content/runs/waste_yolo262/confusion_matrix.png")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# testar modellen på resterande datan vi la undan tidigare

from ultralytics import YOLO

model = YOLO("/content/best.pt")

metrics = model.val(
    data="/content/waste_dataset/data.yaml",
    split="test"
)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
# laddar upp 10 nya bilder

!unzip "OneDrive_2_13-03-2026 (1).zip" -d torture_test

In [ ]:
# använder modellen för att göra tortyrtestet på nya bilder

from ultralytics import YOLO

model = YOLO("/content/best.pt")

results = model.predict(
    source="/content/torture_test",
    conf=0.4,
    save=True
)

In [ ]:
# visar bilderna från testet

from IPython.display import display
from PIL import Image
import os

result_folder = "/content/runs/detect/predict"

for img_name in sorted(os.listdir(result_folder)):
    img_path = os.path.join(result_folder, img_name)
    print(img_name)
    display(Image.open(img_path))

In [ ]:
!zip -r torture_results.zip /content/runs/detect/predict

In [ ]:
from google.colab import files
files.download("torture_results.zip")